In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """read an email from the given address."""
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """send an email to the given address with the given subject and body."""
    return f"Email sent."

In [3]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str

agent = create_agent(
    model="claude-haiku-4-5",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval"
        )
    ]
)

In [4]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "read my email and send a response. You can think of the response on your own."}],
    "email": "Hi Zain, I am going to be late for the meeting tomorrow. Can we reschedule it? Regards, Ash"},
    config=config
)

In [5]:
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'Ash,\n'
                                                                          '\n'
                                                                          'Thanks '
                                                                          'for '
                                                                          'letting '
                                                                          'me '
                                                                          'know '
                                                                          'in '
                                                                          'advance. '
                                                                          'No '
                                                                          'problem '
               

In [6]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': "Hi Ash,\n\nThanks for letting me know in advance. No problem at all—I completely understand that things come up. Let's reschedule the meeting.\n\nWhat day and time work best for you? Feel free to suggest a few options and I'll work with my schedule to accommodate.\n\nLooking forward to connecting soon.\n\nBest regards,\nZain"}, 'description': 'Tool execution requires approval\n\nTool: send_email\nArgs: {\'body\': "Hi Ash,\\n\\nThanks for letting me know in advance. No problem at all—I completely understand that things come up. Let\'s reschedule the meeting.\\n\\nWhat day and time work best for you? Feel free to suggest a few options and I\'ll work with my schedule to accommodate.\\n\\nLooking forward to connecting soon.\\n\\nBest regards,\\nZain"}'}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='8d76fe677c274b689fb92e84306f48a1')]


In [7]:
# access the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi Ash,

Thanks for letting me know in advance. No problem at all—I completely understand that things come up. Let's reschedule the meeting.

What day and time work best for you? Feel free to suggest a few options and I'll work with my schedule to accommodate.

Looking forward to connecting soon.

Best regards,
Zain


In [8]:
# message is approved, so we would like to send it. 
from langgraph.types import Command

response = agent.invoke(
    Command(
        resume = {"decisions": [{"type": "approve"}]}
    ),
    config=config
)

pprint(response)

{'email': 'Hi Zain, I am going to be late for the meeting tomorrow. Can we '
          'reschedule it? Regards, Ash',
 'messages': [HumanMessage(content='read my email and send a response. You can think of the response on your own.', additional_kwargs={}, response_metadata={}, id='9c838680-6693-4cb2-bfc9-4fe271a218a8'),
              AIMessage(content=[{'text': "I'll read your email first and then compose an appropriate response.", 'type': 'text'}, {'id': 'toolu_01YEZT2o8svtzmd9d5mi3vUx', 'input': {}, 'name': 'read_email', 'type': 'tool_use', 'caller': {'type': 'direct'}}], additional_kwargs={}, response_metadata={'id': 'msg_01XsU7GLjC12nFDXQzMeTPui', 'model': 'claude-haiku-4-5-20251001', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'input_tokens': 623, 'output_tokens': 50, 'server_tool_use': None, 'service_tier': 'standard',